# 01.2 Tensor Shape Ops

The core task of this notebook is to build genuine shape thinking.

Many `PyTorch` bugs are not mathematical errors, but shape errors.

Key concepts:

- reshape
- flatten
- unsqueeze and squeeze
- transpose
- permute
- broadcasting

## Learning Goals

After this notebook, you should be able to:

1. Use `reshape`, `view`, and `flatten` comfortably.
2. Understand `unsqueeze` and `squeeze`.
3. Distinguish `transpose` from `permute`.
4. Predict output shapes after common operations.
5. Explain broadcasting using shapes.
6. Debug shape mismatch errors faster.

In [ ]:
import torch

## `reshape`, `view`, and `flatten`

They all describe how the external shape of a tensor changes.

- the most commonly used and usually convenient
- requires compatible memory layout
- collapses dimensions into one

In [ ]:
x = torch.arange(24)
a = x.reshape(2, 3, 4)
b = a.flatten()
c = a.flatten(start_dim=1)

print("x.shape =", x.shape)
print("a.shape =", a.shape)
print("b.shape =", b.shape)
print("c.shape =", c.shape)

`flatten(start_dim=1)` is very common in neural networks.

For example:

- input is `(batch, channels, height, width)`
- after flattening it becomes `(batch, features)`

This usually happens before feeding convolution outputs into linear layers.


In [ ]:
# Exercise 1
# Given x.shape == (2, 3, 4)
#1. reshape it to (6, 4)
# 2. flatten it to (2, 12)

x = torch.arange(24).reshape(2, 3, 4)

# x1 =
# x2 =
# print(x1.shape)
# print(x2.shape)

In [ ]:
# Exercise 1 Reference Solution

x = torch.arange(24).reshape(2, 3, 4)
x1 = x.reshape(6, 4)
x2 = x.flatten(start_dim=1)
print(x1.shape)
print(x2.shape)

## `unsqueeze` and `squeeze`

These two operations add or remove dimensions of size 1.

- insert a size-1 dimension
- remove a size-1 dimension

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])
x_row = x.unsqueeze(0)
x_col = x.unsqueeze(1)
x_back = x_col.squeeze(1)

print("x.shape =", x.shape)
print("x_row.shape =", x_row.shape)
print("x_col.shape =", x_col.shape)
print("x_back.shape =", x_back.shape)

These operations commonly appear when:

- manually creating a batch dimension
- controlling broadcasting direction
- matching a layer's input interface

In [ ]:
# Exercise 2
# Given x.shape == (4,)
#1. turn it into (1, 4)
# 2. turn it into (4, 1)
# 3. then turn (4, 1) turn back into (4,)

x = torch.arange(4)

# a =
# b =
# c =
# print(a.shape, b.shape, c.shape)

In [ ]:
# Exercise 2 Reference Solution

x = torch.arange(4)
a = x.unsqueeze(0)
b = x.unsqueeze(1)
c = b.squeeze(1)
print(a.shape, b.shape, c.shape)

## `transpose` and `permute`

Both operations reorder dimensions, but at different levels of flexibility.

- swap two dimensions
- reorder all dimensions by a specified order

In [ ]:
x = torch.arange(24).reshape(2, 3, 4)
xt = x.transpose(1, 2)
xp = x.permute(2, 0, 1)

print("x.shape =", x.shape)
print("xt.shape =", xt.shape)
print("xp.shape =", xp.shape)

Common image shape pattern:

- `PyTorch` commonly use `(batch, channels, height, width)`
- some external libraries often use `(height, width, channels)`

This is one reason why `permute` appears so often.


In [ ]:
# Exercise 3
# Given image.shape == (3, 32, 32), meaning (channels, height, width)
#Convert it to (height, width, channels)
# Convert it to (height, width, channels)

image = torch.randn(3, 32, 32)

# image_hwc =
# print(image_hwc.shape)

In [ ]:
# Exercise 3 Reference Solution

image = torch.randn(3, 32, 32)
image_hwc = image.permute(1, 2, 0)
print(image_hwc.shape)

## Broadcasting

Broadcasting is not magic auto-expansion; it is dimension alignment.

Minimal rule:

- compare dimensions from the end
- dimensions must match, or one of them must be 1

In [ ]:
x = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
offset = torch.tensor([10.0, 100.0])
row_scale = torch.tensor([[1.0], [10.0], [100.0]])

print("x + offset =\n", x + offset)
print()
print("x * row_scale =\n", x * row_scale)

How to read it:

- `x.shape == (3, 2)`
- broadcasts row-wise

In [ ]:
# Exercise 4
# Implement center_by_column(x) so each column subtracts its own mean.

def center_by_column(x):
    # TODO
    pass


# sample = torch.tensor([[1.0, 10.0], [2.0, 20.0], [3.0, 30.0]])
# print(center_by_column(sample))

In [ ]:
# Exercise 4 Reference Solution

def center_by_column_solution(x):
    col_mean = x.mean(dim=0, keepdim=True)
    return x - col_mean


sample = torch.tensor([[1.0, 10.0], [2.0, 20.0], [3.0, 30.0]])
print(center_by_column_solution(sample))

## Non-Contiguous Tensors

This is a practical but often overlooked topic.

After operations like `transpose` or `permute`, a tensor may become non-contiguous.

In such cases:

- `reshape` is usually safer
- `view` may fail

In [ ]:
x = torch.arange(24).reshape(2, 3, 4)
xt = x.transpose(1, 2)

print("xt.is_contiguous() =", xt.is_contiguous())

try:
    bad = xt.view(2, 12)
    print("view result shape =", bad.shape)
except RuntimeError as e:
    print("view failed / view failed:", e)

good = xt.reshape(2, 12)
print("reshape result shape =", good.shape)

In [ ]:
# Exercise 5
# Given x.shape == (2, 3, 4)
#1. First transpose to (2, 4, 3)
# 2. Then reshape to (2, 12)

x = torch.arange(24).reshape(2, 3, 4)

# y =
# z =
# print(y.shape)
# print(z.shape)

In [ ]:
# Exercise 5 Reference Solution

x = torch.arange(24).reshape(2, 3, 4)
y = x.transpose(1, 2)
z = y.reshape(2, 12)
print(y.shape)
print(z.shape)

## Summary

You should now start building a fixed habit:

1. first write down the input shape
2. then predict the output shape
3. finally run the code to verify

You should now be able to answer:

1. What problems do `reshape`, `flatten`, and `unsqueeze` solve?
2. What is the difference between `transpose` and `permute`?
3. Why does `view` sometimes fail while `reshape` works?
4. What is the minimal rule for broadcasting?

Suggested next step:

- Move to `01_03_autograd.ipynb` to understand how gradients are tracked and propagated.